<a href="https://colab.research.google.com/github/MarianoVIsabella/Data-Warehouse-Project/blob/main/DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_PATH = '/content/drive/MyDrive'

In [3]:
!pip install import_ipynb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.2 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import warnings
from datetime import datetime
import os
import import_ipynb
os.chdir(BASE_PATH + '/Colab Notebooks')
from DataQuality import DQAReport, parse_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
np.random.seed(42)
df_match_start = pd.read_csv(BASE_PATH + "/matches_1930_2022.csv",  dtype={'home_penalty': 'Int64', 'away_penalty': 'Int64'})
#Int64 is a NULLABLE datatype, so that the integer value can stay int, othervise pandas convert them in float to represent null.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.3/400.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.8 MB/s eta 0:00:00


In [5]:
#DATA QUALITY VARIABLES AND RULES
key_fields=['home_team', 'away_team', 'Date'] #columns composing the primary key
#columns that should NOT have any null value
required=['home_team', 'away_team', 'home_score', 'away_score', 'home_manager', 'away_manager','Attendance', 'Round', 'Date', 'Year']
#Compute the oldest date that should be tolerated in the dataset (1930, year of the first world Cup Edition)
current_year= datetime.today().year
range_len= (current_year - 1930) + 1 #the +1 guarantees that all dates from the very first year are always taken.

match_validity_rules = {
    'home_score' : lambda s: s >= 0,
    'away_score' : lambda s: s >= 0,
    'Attendance' : lambda s: s.between(1, 200000),
    'home_xg' : lambda s: s.between (0, 12),
    'away_xg' : lambda s: s.between (0, 12),
    'home_penalty' : lambda s: s >= 0,
    'away_penalty' : lambda s: s >= 0,
    'Year': lambda s: s.isin([1930,1934,1938] + [x for x in range(1950,current_year,4)]), #check the year is one of the real year of World Cup editions
                                                                                          #(every 4 year starting from 1930, excluded 1942 and 1946)
    'Round': lambda s: s.isin(['Second round', 'Round of 16', 'Quarter-finals', 'Second group stage', 'First group stage', 'Group stage', 'Final stage', 'Semi-finals', 'First round', 'Third-place match', 'Final', 'Group stage play-off']),
    'Date' : lambda s: pd.to_datetime(s, errors='coerce').notna(), # All dates should be in a valid format
    'Score' : lambda s: s == s.str.strip() #No space at the beginning or at the end of the value
}

match_consistency_rules = [
    lambda df: (lambda hp, h, a, ap: (
        (df['home_score'] == h) & (df['away_score'] == a)
    ))(*parse_score(df['Score'])),
    lambda df: ~(df['home_penalty'].notna()) | (lambda hp, h, a, ap: (
        df['home_penalty'] == hp
    ))(*parse_score(df['Score'])),
    lambda df: ~(df['away_penalty'].notna()) | (lambda hp, h, a, ap: (
        df['away_penalty'] == ap
    ))(*parse_score(df['Score'])),
    lambda df: df['home_penalty'].notna() == df['away_penalty'].notna(),
    lambda df: ~(df['home_penalty'] > 0) | df['home_penalty_shootout_goal_long'].notna(),
    lambda df: ~(df['away_penalty'] > 0) | df['away_penalty_shootout_goal_long'].notna(),
    lambda df: ~(df['home_penalty'] == 0) | df['home_penalty_shootout_miss_long'].notna(),
    lambda df: ~(df['away_penalty'] == 0) | df['away_penalty_shootout_miss_long'].notna(),
    lambda df: df['home_xg'].notna() == df['away_xg'].notna(),
    lambda df: df['Year'] == pd.to_datetime(df['Date'], errors = 'coerce').dt.year,
    lambda df: ~(df['home_score'] > 0) | ((df['home_goal'].notna() & df['home_goal_long'].notna()) | (df['home_penalty_goal'].notna()) | (df['home_own_goal'].notna())),
    lambda df: ~(df['away_score'] > 0) | ((df['away_goal'].notna() & df['away_goal_long'].notna()) | (df['away_penalty_goal'].notna()) | (df['away_own_goal'].notna())),
    lambda df: df['home_team'] != df['away_team'],
    lambda df: df['home_captain'] != df['away_captain'],
    lambda df: df['home_manager'] != df['away_manager']
]

# Data Cleaning Classes

In [6]:
class AuditLog:
    """
    Records every cleaning transformation applied to the data.
    Each entry captures: step name, column, row index, before, after, timestamp.
    """
    def __init__(self):
        self._entries = []

    def log(self, step: str, col: str, idx, before, after, reason: str = ''):

        self._entries.append({
            'step'      : step,
            'column'    : col,
            'row_index' : idx,
            'before'    : before,
            'after'     : after,
            'reason'    : reason,
            'timestamp' : datetime.now().isoformat()
        })

    def log_batch(self, step: str, col: str, mask: pd.Series,
                  before_series: pd.Series, after_series: pd.Series, reason: str = ''):
        changed_idx = mask[mask].index
        for idx in changed_idx:
            self.log(step, col, idx,
                     str(before_series.get(idx, 'N/A')),
                     str(after_series.get(idx, 'N/A')),
                     reason)

    def to_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._entries)

    def summary(self) -> pd.DataFrame:
        if not self._entries:
            return pd.DataFrame()
        df = self.to_df()
        return (df.groupby('step')
                  .agg(changes=('row_index','count'),
                       cols_affected=('column', lambda x: ', '.join(x.unique())))
                  .reset_index()
                  .sort_values('changes', ascending=False))

    def __len__(self):
        return len(self._entries)

In [7]:
#Utility function we need in the pipeline
def _map(val, reverse, unknown_value=None):
  if pd.isna(val): return val
  key = str(val).strip().lower()
  if key in reverse:
    return reverse[key]
  return unknown_value if unknown_value is not None else val

In [8]:
"""
Class responsible of the pipeline: it has methods for replacing and standardizing string values, mapping everything in a audit log
"""
class CleaningPipeline:

    def __init__(self, df: pd.DataFrame, table_name: str):
        self.original   = df.copy()
        self.df         = df.copy()
        self.table_name = table_name
        self.audit      = AuditLog()
        self._steps_run = []

    def standardize_strings(self, cols: list,
                              strip: bool = True,
                              lower: bool = False,
                              title_case: bool = False):

        for col in cols:
            if col not in self.df.columns: continue
            before = self.df[col].copy()
            s = self.df[col].astype(str)
            if strip:      s = s.str.strip()
            if lower:      s = s.str.lower()
            if title_case: s = s.str.title()
            changed = (s != before.astype(str)) & before.notna()
            self.df.loc[changed, col] = s[changed]
            self.audit.log_batch('standardize_strings', col, changed,
                                  before, self.df[col],
                                  'Strip whitespace + normalize case')

        self._steps_run.append('standardize_strings')
        return self

    def canonicalize_enum(self, col: str, mapping: dict,
                        unknown_value: str = None):

      if col not in self.df.columns: return self
      reverse = {}
      for canonical, variants in mapping.items():
          for v in variants:
              reverse[v.strip().lower()] = canonical
      before = self.df[col].copy()

      self.df[col] = self.df[col].apply(_map, args=(reverse,unknown_value))
      changed = (self.df[col] != before) & before.notna()
      self.audit.log_batch('canonicalize_enum', col, changed,
                            before, self.df[col],
                            f'Canonical mapping for {col}')
      self._steps_run.append(f'canonicalize_enum:{col}')

      return self

    def summary(self):
        orig_rows = len(self.original)
        curr_rows = len(self.df)
        print(f'\n{"═"*55}')
        print(f'  CLEANING PIPELINE SUMMARY — {self.table_name}')
        print(f'{"═"*55}')
        print(f'  Original rows  : {orig_rows}')
        print(f'  Current rows   : {curr_rows} ({orig_rows - curr_rows} removed)')
        print(f'  Steps run      : {len(self._steps_run)}')
        print(f'  Audit entries  : {len(self.audit)}')
        print(f'{"─"*55}')
        print(self.audit.summary().to_string(index=False))
        print(f'{"═"*55}')

    @property
    def clean_df(self) -> pd.DataFrame:
        return self.df.copy()


# Data Cleaning tasks


1.   The very first thing we do is to ensure all the text fields in the dataset doesn't begin or end with spaces. This step will solve the validity issue on the Score column.

2.    After this, some of the nation name were updated to match historical changes. However, not every "former name" was changed, with the following changes performed:
*   Zaire -> DR. Congo
*   Dutch East Indies -> Indonesia
*   Czech Republic -> Czechia
*   Serbia and Montenegro -> Serbia  
*   FR Yugoslavia -> Serbia  
The motivation of every choice is explained in the report

3.  About round names, the following considerations and decision are taken (they are motivated in the report) :


*   Final Stage stays the same.
*   Fist Group Stage -> Group Stage.
*   First Round -> Group Stage.
*   Second Group Stage stays the same.
*   Second Round -> Second Group Stage.
*   Group stage play-off stays the same.




In [9]:
textual_fields=['Score','home_team','away_team','home_manager','away_manager','home_captain','away_captain','Venue',
                'Officials','Round','Referee','Host','Notes','home_goal','away_goal','home_goal_long','away_goal_long',
                'home_own_goal','away_own_goal','home_penalty','away_penalty','home_penalty_goal','away_penalty_goal',
                'home_penalty_miss_long','away_penalty_miss_long','home_penalty_shootout_goal_long','away_penalty_shootout_goal_long',
                'home_penalty_shootout_miss_long','away_penalty_shootout_miss_long','home_red_card','away_red_card',
                'home_yellow_red_card','away_yellow_red_card','home_yellow_card_long','away_yellow_card_long','home_substitute_in_long',
                'away_substitute_in_long']
nations_map = {
    'DR. Congo' : ['Zaire'], #to actualize rows, update Zaire's state name to DR. Congo
    'Czechia': ['Czech Republic'], #also this is a change to update the state name
    'Indonesia': ['Dutch East Indies'], #another name change, because the Dutch East Indies were in fact a colonial state
    'Serbia': ['Serbia','Serbia and Montenegro', 'FR Yugoslavia']
    #After 1992, FR Yugoslavia was one of the States born from Yugoslavia. It was composed of Serbia and Montenegro and in 2003 it changed name in 'Serbia and Montenegro'
    #After 2006 referendum, Montenegro became an indipendent country and Serbia inherited 'Serbia and Montenegro' sports title
}
#This map ensures the same phases of the tournament always had the same name during time
round_map = {
    'Group stage' : ['Group stage','First group stage', 'First round'],
    'Second group stage': ['Second group stage','Second round']
}

pipe_prod = (
    CleaningPipeline(df_match_start, 'match_fact')
    .standardize_strings(textual_fields, strip=True, title_case=False)
    .canonicalize_enum('home_team', nations_map)
    .canonicalize_enum('away_team', nations_map)
    .canonicalize_enum('Round',round_map)
)

pipe_prod.summary()
clean_products_output_path = BASE_PATH + '/matches_clean.csv'
products_audit_output_path = BASE_PATH + '/matches_audit_log.csv'
products_audit_summary_output_path = BASE_PATH + '/matches_audit_summary.csv'
pipe_prod.clean_df.to_csv(clean_products_output_path, index=False)
pipe_prod.audit.to_df().to_csv(products_audit_output_path, index=False)
pipe_prod.audit.summary().to_csv(products_audit_summary_output_path, index=False)


═══════════════════════════════════════════════════════
  CLEANING PIPELINE SUMMARY — match_fact
═══════════════════════════════════════════════════════
  Original rows  : 964
  Current rows   : 964 (0 removed)
  Steps run      : 4
  Audit entries  : 123
───────────────────────────────────────────────────────
               step  changes               cols_affected
  canonicalize_enum      122 home_team, away_team, Round
standardize_strings        1                       Score
═══════════════════════════════════════════════════════


In [10]:
clean_match= pd.read_csv(BASE_PATH + '/matches_clean.csv')

dqa_match =(
    DQAReport(clean_match, table_name='match_fact', primary_key=key_fields)
    .check_completeness(required_cols=required)
    .check_uniqueness(key_cols=key_fields)
    .check_validity(match_validity_rules)
    .check_consistency(match_consistency_rules)
    .check_timeliness('Date', max_age_days=365 * range_len, allow_future=False)
)
scorecard_match = dqa_match.scorecard()
scorecard_match.to_csv(BASE_PATH + '/dq_scorecard_match_clean.csv', index=False)

dqa_match.display_scorecard(
    scorecard_match[['Dimension', 'Score', 'Issues', 'Status', 'Details']],
    'MATCH DATASET',
    dqa_match.overall_score()
)

Dimension,Score,Issues,Status,Details
Completeness,100.00%,0,🟢,Missing per column: {}
Uniqueness,100.00%,0,🟢,"Duplicate rows on ['home_team', 'away_team', 'Date']: 0"
Validity,100.00%,0,🟢,home_score: 0 invalid | away_score: 0 invalid | Attendance: 0 invalid | home_xg: 0 invalid | away_xg: 0 invalid | home_penalty: 0 invalid | away_penalty: 0 invalid | Year: 0 invalid | Round: 0 invalid | Date: 0 invalid | Score: 0 invalid
Consistency,100.00%,0,🟢,rule_0: 0 violations | rule_1: 0 violations | rule_2: 0 violations | rule_3: 0 violations | rule_4: 0 violations | rule_5: 0 violations | rule_6: 0 violations | rule_7: 0 violations | rule_8: 0 violations | rule_9: 0 violations | rule_10: 0 violations | rule_11: 0 violations | rule_12: 0 violations | rule_13: 0 violations | rule_14: 0 violations
Timeliness,100.00%,0,🟢,Future dates: 0 | Stale (>35405d): 0
